In [ ]:

from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd

USE_COLS: List[str] = [
    "INSTRUMENT", "SYMBOL", "EXPIRY_DT", "STRIKE_PR", "OPTION_TYP",
    "OPEN", "HIGH", "LOW", "CLOSE", "SETTLE_PR", "CONTRACTS",
    "VAL_INLAKH", "OPEN_INT", "CHG_IN_OI", "TIMESTAMP",
]

NUMERIC_COLS: List[str] = [
    "STRIKE_PR", "OPEN", "HIGH", "LOW", "CLOSE", "SETTLE_PR",
    "CONTRACTS", "VAL_INLAKH", "OPEN_INT", "CHG_IN_OI",
]

TEXT_COLS: List[str] = ["INSTRUMENT", "SYMBOL", "OPTION_TYP"]

OUTLIER_COLS: List[str] = [
    "OPEN", "HIGH", "LOW", "CLOSE", "SETTLE_PR",
    "CONTRACTS", "OPEN_INT", "VAL_INLAKH",
]

GROUP_COLS: List[str] = ["SYMBOL", "EXPIRY_DT", "STRIKE_PR", "OPTION_TYP"]

FEATURE_COLUMNS: List[str] = [
    "STRIKE_PR", "OPEN", "HIGH", "LOW", "CLOSE", "SETTLE_PR",
    "CONTRACTS", "OPEN_INT", "CHG_IN_OI",
    "days_to_expiry", "price_range", "open_close_gap",
    "oi_change_ratio", "value_per_contract",
    "prev_close", "prev_close_2", "prev_close_3",
    "price_momentum_1", "price_momentum_2",
    "roll_mean_close_3", "roll_std_close_3",
    "roll_mean_close_5", "roll_std_close_5",
    "roll_mean_oi_3",    "roll_mean_oi_5",
    "rsi", "volatility_ratio", "oi_trend",
    "is_call", "moneyness",
]

class DataLoader:
    
    def __init__(
        self,
        csv_path: str | Path,
        symbols: Optional[List[str]] = None,
        max_rows: int = 50_000,
        chunksize: int = 100_000,
        max_chunks: int = 12,
        random_state: int = 42,
    ):
        self.csv_path = Path(csv_path)
        self.symbols = {s.upper() for s in (symbols or ["NIFTY", "BANKNIFTY"])}
        self.max_rows = max_rows
        self.chunksize = chunksize
        self.max_chunks = max_chunks
        self.random_state = random_state

    def load(self) -> pd.DataFrame:
        sample_per_chunk = max(1_000, self.max_rows // self.max_chunks)
        pieces: List[pd.DataFrame] = []

        for chunk_num, chunk in enumerate(
            pd.read_csv(
                self.csv_path, usecols=USE_COLS,
                chunksize=self.chunksize, low_memory=False,
            ),
            start=1,
        ):
            chunk = self._clean_chunk(chunk)
            if chunk.empty:
                if chunk_num >= self.max_chunks:
                    break
                continue

            take = min(sample_per_chunk, len(chunk))
            pieces.append(chunk.sample(n=take, random_state=self.random_state + chunk_num))

            if chunk_num >= self.max_chunks:
                break

        if not pieces:
            raise ValueError(f"No usable rows found in '{self.csv_path}'.")

        data = pd.concat(pieces, ignore_index=True)
        if len(data) > self.max_rows:
            data = data.sample(n=self.max_rows, random_state=self.random_state).reset_index(drop=True)

        return data


    def _clean_chunk(self, chunk: pd.DataFrame) -> pd.DataFrame:
        chunk = self._standardize_text(chunk)
        chunk = chunk[chunk["SYMBOL"].isin(self.symbols)]
        if chunk.empty:
            return chunk
        chunk = self._convert_numeric(chunk)
        chunk = self._parse_dates(chunk)
        chunk = self._apply_validity_filters(chunk)
        return chunk

    def _standardize_text(frame: pd.DataFrame) -> pd.DataFrame:
        frame = frame.copy()
        for col in TEXT_COLS:
            frame[col] = frame[col].astype(str).str.strip().str.upper()
        return frame

    def _convert_numeric(frame: pd.DataFrame) -> pd.DataFrame:
        frame = frame.copy()
        for col in NUMERIC_COLS:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
        return frame

    def _parse_dates(frame: pd.DataFrame) -> pd.DataFrame:
        frame = frame.copy()
        for col in ("TIMESTAMP", "EXPIRY_DT"):
            frame[col] = pd.to_datetime(
                frame[col].astype(str).str.strip().str.title(),
                format="%d-%b-%Y", errors="coerce",
            )
        return frame

    def _apply_validity_filters(frame: pd.DataFrame) -> pd.DataFrame:
        frame = frame.copy()

        frame = frame[frame["INSTRUMENT"].isin(["OPTIDX", "OPTSTK"])]
        frame = frame[frame["OPTION_TYP"].isin(["CE", "PE"])]

        frame = frame[frame["STRIKE_PR"] > 0]
        frame = frame.dropna(subset=NUMERIC_COLS + ["TIMESTAMP", "EXPIRY_DT"])

        non_neg = ["STRIKE_PR", "OPEN", "HIGH", "LOW", "CLOSE",
                   "SETTLE_PR", "CONTRACTS", "VAL_INLAKH", "OPEN_INT"]
        frame = frame[(frame[non_neg] >= 0).all(axis=1)]

        frame = frame[frame["HIGH"] >= frame["LOW"]]

        frame = frame[frame["EXPIRY_DT"] >= frame["TIMESTAMP"]]

        return frame

class OutlierTrimmer:

    def __init__(self, columns: List[str] = OUTLIER_COLS, lower: float = 0.01, upper: float = 0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper
        self._bounds: dict[str, Tuple[float, float]] = {}

    def fit(self, frame: pd.DataFrame) -> "OutlierTrimmer":
        for col in self.columns:
            self._bounds[col] = (
                frame[col].quantile(self.lower),
                frame[col].quantile(self.upper),
            )
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        if not self._bounds:
            raise RuntimeError("Call .fit() before .transform().")
        frame = frame.copy()
        for col, (low, high) in self._bounds.items():
            frame = frame[frame[col].between(low, high)]
        return frame

    def fit_transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        return self.fit(frame).transform(frame)

class FeatureEngineer:
    def _rolling(frame: pd.DataFrame, col: str, window: int, stat: str) -> pd.Series:
        return frame.groupby(GROUP_COLS)[col].transform(
            lambda s: getattr(s.rolling(window, min_periods=1), stat)()
        )

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        frame = frame.copy()
        frame = frame.sort_values(GROUP_COLS + ["TIMESTAMP"]).reset_index(drop=True)

        frame = self._add_basic_features(frame)
        frame = self._add_lag_features(frame)
        frame = self._add_rolling_features(frame)
        frame = self._add_rsi(frame)
        frame = self._add_volatility_and_oi_trend(frame)
        frame = self._add_option_features(frame)
        frame = self._add_targets(frame)

        required = ["prev_close", "prev_close_2", "next_close"]
        frame = frame.dropna(subset=required)

        return frame

    def _add_basic_features(frame: pd.DataFrame) -> pd.DataFrame:
        frame["days_to_expiry"] = (frame["EXPIRY_DT"] - frame["TIMESTAMP"]).dt.days
        frame["price_range"] = frame["HIGH"] - frame["LOW"]
        frame["open_close_gap"] = frame["CLOSE"] - frame["OPEN"]
        frame["oi_change_ratio"] = np.where(
            frame["OPEN_INT"] == 0, 0.0,
            frame["CHG_IN_OI"] / frame["OPEN_INT"],
        )
        frame["value_per_contract"] = np.where(
            frame["CONTRACTS"] == 0, 0.0,
            frame["VAL_INLAKH"] / frame["CONTRACTS"],
        )
        return frame

    def _add_lag_features(frame: pd.DataFrame) -> pd.DataFrame:
        g = frame.groupby(GROUP_COLS)["CLOSE"]
        frame["prev_close"]     = g.shift(1)
        frame["prev_close_2"]   = g.shift(2)
        frame["prev_close_3"]   = g.shift(3)
        frame["price_momentum_1"] = frame["CLOSE"] - frame["prev_close"]
        frame["price_momentum_2"] = frame["CLOSE"] - frame["prev_close_2"]
        return frame

    def _add_rolling_features(self, frame: pd.DataFrame) -> pd.DataFrame:
        for w in (3, 5):
            frame[f"roll_mean_close_{w}"] = self._rolling(frame, "CLOSE",    w, "mean")
            frame[f"roll_std_close_{w}"]  = self._rolling(frame, "CLOSE",    w, "std").fillna(0)
            frame[f"roll_mean_oi_{w}"]    = self._rolling(frame, "OPEN_INT", w, "mean")
        return frame

    def _add_rsi(frame: pd.DataFrame, period: int = 14) -> pd.DataFrame:
        delta = frame.groupby(GROUP_COLS)["CLOSE"].diff()
        gain = delta.clip(lower=0)
        loss = (-delta).clip(lower=0)

        avg_gain = (
            frame.assign(_g=gain)
            .groupby(GROUP_COLS)["_g"]
            .transform(lambda s: s.rolling(period, min_periods=1).mean())
        )
        avg_loss = (
            frame.assign(_l=loss)
            .groupby(GROUP_COLS)["_l"]
            .transform(lambda s: s.rolling(period, min_periods=1).mean())
        )

        rs = avg_gain / avg_loss.replace(0, np.nan)
        frame["rsi"] = (100 - (100 / (1 + rs))).fillna(50)  
        return frame

    def _add_volatility_and_oi_trend(self, frame: pd.DataFrame) -> pd.DataFrame:
        frame["volatility_ratio"] = np.where(
            frame["roll_mean_close_5"] == 0, 0.0,
            frame["roll_std_close_5"] / frame["roll_mean_close_5"].abs(),
        )
        frame["oi_trend"] = (
            frame.groupby(GROUP_COLS)["OPEN_INT"]
            .transform(lambda s: s.pct_change(periods=3).fillna(0))
        )
        return frame

    def _add_option_features(frame: pd.DataFrame) -> pd.DataFrame:
        frame["is_call"] = (frame["OPTION_TYP"] == "CE").astype(int)

        frame["moneyness"] = np.where(
            frame["SETTLE_PR"] == 0, 0.0,
            (frame["STRIKE_PR"] - frame["SETTLE_PR"]) / frame["SETTLE_PR"],
        )
        return frame

    def _add_targets(frame: pd.DataFrame) -> pd.DataFrame:
        g = frame.groupby(GROUP_COLS)["CLOSE"]
        frame["next_close"] = g.shift(-1)
        frame["target_direction"] = (frame["next_close"] > frame["CLOSE"]).astype(int)
        return frame

class TimeBasedSplitter:
    def __init__(self, train_ratio: float = 0.70, val_ratio: float = 0.10):
        if not (0 < train_ratio < 1 and 0 < val_ratio < 1):
            raise ValueError("Ratios must be strictly between 0 and 1.")
        if train_ratio + val_ratio >= 1.0:
            raise ValueError("train_ratio + val_ratio must be < 1.")
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio

    def split(
        self, frame: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        frame = frame.sort_values(
            ["TIMESTAMP", "SYMBOL", "EXPIRY_DT", "STRIKE_PR", "OPTION_TYP"]
        ).reset_index(drop=True)

        dates = frame["TIMESTAMP"].drop_duplicates().sort_values().tolist()
        n = len(dates)

        if n >= 3:
            train_cut_idx = max(1, int(n * self.train_ratio)) - 1
            val_cut_idx   = max(train_cut_idx + 1, int(n * (self.train_ratio + self.val_ratio))) - 1

            train_cut = dates[train_cut_idx]
            val_cut   = dates[val_cut_idx]

            train = frame[frame["TIMESTAMP"] <= train_cut].copy()
            val   = frame[(frame["TIMESTAMP"] > train_cut) & (frame["TIMESTAMP"] <= val_cut)].copy()
            test  = frame[frame["TIMESTAMP"] > val_cut].copy()
        else:
            n_rows = len(frame)
            t_end = int(n_rows * self.train_ratio)
            v_end = int(n_rows * (self.train_ratio + self.val_ratio))
            train = frame.iloc[:t_end].copy()
            val   = frame.iloc[t_end:v_end].copy()
            test  = frame.iloc[v_end:].copy()

        return train, val, test


class PreprocessingPipeline:
    def __init__(
        self,
        csv_path: str | Path,
        symbols: Optional[List[str]] = None,
        max_rows: int = 50_000,
        train_ratio: float = 0.70,
        val_ratio: float = 0.10,
        random_state: int = 42,
    ):
        self.csv_path = csv_path
        self.symbols = symbols
        self.max_rows = max_rows
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.random_state = random_state

        self._loader   = DataLoader(csv_path, symbols, max_rows, random_state=random_state)
        self._trimmer  = OutlierTrimmer()
        self._engineer = FeatureEngineer()
        self._splitter = TimeBasedSplitter(train_ratio, val_ratio)

        self.train: Optional[pd.DataFrame] = None
        self.val:   Optional[pd.DataFrame] = None
        self.test:  Optional[pd.DataFrame] = None

    def run(self, verbose: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

        df = self._loader.load()
        self._log(verbose, f"Rows after loading       : {len(df):,}")

        df = self._trimmer.fit_transform(df)
        self._log(verbose, f"Rows after trimming      : {len(df):,}")

        df = self._engineer.transform(df)
        self._log(verbose, f"Rows after feature build : {len(df):,}")

        self.train, self.val, self.test = self._splitter.split(df)

        self._log(verbose, f"Training rows            : {len(self.train):,}")
        self._log(verbose, f"Validation rows          : {len(self.val):,}")
        self._log(verbose, f"Test rows                : {len(self.test):,}")

        self._log(verbose, self._class_balance_summary())

        return self.train, self.val, self.test

    def save(self, output_dir: str | Path) -> None:
        if self.train is None:
            raise RuntimeError("Run .run() before .save().")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        self.train.to_parquet(out / "train.parquet", index=False)
        self.val.to_parquet(out / "val.parquet",   index=False)
        self.test.to_parquet(out / "test.parquet",  index=False)
        print(f"Splits saved to '{out}/'")

    def load(input_dir: str | Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
      
        d = Path(input_dir)
        train = pd.read_parquet(d / "train.parquet")
        val   = pd.read_parquet(d / "val.parquet")
        test  = pd.read_parquet(d / "test.parquet")
        print(f"Splits loaded from '{d}/'  "
              f"(train={len(train):,}  val={len(val):,}  test={len(test):,})")
        return train, val, test

    def _class_balance_summary(self) -> str:
        if self.train is None:
            return ""
        up = self.train["target_direction"].mean() * 100
        return f"Train class balance      : UP={up:.1f}%  DOWN/FLAT={100-up:.1f}%"

    def _log(verbose: bool, msg: str) -> None:
        if verbose:
            print(msg)

if __name__ == "__main__":
    import sys

    csv = sys.argv[1] if len(sys.argv) > 1 else "fobhav_noisy.csv"
    out = sys.argv[2] if len(sys.argv) > 2 else "processed"

    pipe = PreprocessingPipeline(csv_path=csv)
    train, val, test = pipe.run()
    pipe.save(out)

    print("\nFeature columns available:")
    for fc in FEATURE_COLUMNS:
        print(f"  {fc}")